In [38]:
import json
import os
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer, make_column_selector as selector
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [39]:
RANDOM_STATE = 42
TEST_SIZE = 0.2
MODEL_DIR = Path("artifacts")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

EVAL_PATH = Path("app/db/extrait_eval.csv")
SONDAGE_PATH = Path("app/db/extrait_sondage.csv")
SIRH_PATH = Path("app/db/extrait_sirh.csv")

TARGET_COL = "a_quitte_l_entreprise"
MODEL_FILENAME = "attrition_pipeline.joblib"
METADATA_FILENAME = "model_metadata.json"

In [40]:
eval_df = pd.read_csv(EVAL_PATH)
sondage_df = pd.read_csv(SONDAGE_PATH)
sirh_df = pd.read_csv(SIRH_PATH)

print(eval_df.shape, sondage_df.shape, sirh_df.shape)

(1470, 10) (1470, 12) (1470, 12)


In [41]:
def normalize_yes_no(s: pd.Series) -> pd.Series:
    return (
        s.astype(str)
         .str.strip()
         .str.lower()
         .replace({
             "yes": "oui",
             "no": "non",
             "y": "oui",
             "n": "non",
             "true": "oui",
             "false": "non"
         })
    )

def assert_unique_key(df: pd.DataFrame, key: str, name: str) -> None:
    if key not in df.columns:
        raise KeyError(f"{name}: clé absente -> {key}")
    if df[key].duplicated().any():
        raise ValueError(f"{name}: doublons détectés sur la clé {key}")

In [42]:
print(eval_df.columns)
print(sondage_df.columns)
print(sirh_df.columns)

Index(['satisfaction_employee_environnement', 'note_evaluation_precedente',
       'niveau_hierarchique_poste', 'satisfaction_employee_nature_travail',
       'satisfaction_employee_equipe',
       'satisfaction_employee_equilibre_pro_perso', 'eval_number',
       'note_evaluation_actuelle', 'heure_supplementaires',
       'augementation_salaire_precedente'],
      dtype='object')
Index(['a_quitte_l_entreprise', 'nombre_participation_pee',
       'nb_formations_suivies', 'nombre_employee_sous_responsabilite',
       'code_sondage', 'distance_domicile_travail', 'niveau_education',
       'domaine_etude', 'ayant_enfants', 'frequence_deplacement',
       'annees_depuis_la_derniere_promotion', 'annes_sous_responsable_actuel'],
      dtype='object')
Index(['id_employee', 'age', 'genre', 'revenu_mensuel', 'statut_marital',
       'departement', 'poste', 'nombre_experiences_precedentes',
       'nombre_heures_travailless', 'annee_experience_totale',
       'annees_dans_l_entreprise', 'annees_

In [43]:
# --- eval_df ---
# enlever le préfixe "E_" dans la colonne clé
eval_df["eval_number"] = eval_df["eval_number"].str.replace("E_", "", regex=False).astype(int)

# renommer la colonne
eval_df = eval_df.rename(columns={"eval_number": "id_employee"})


# --- sondage_df ---
sondage_df = sondage_df.rename(columns={"code_sondage": "id_employee"})

In [44]:
def build_training_dataframe(
    sirh_df: pd.DataFrame,
    eval_df: pd.DataFrame,
    sondage_df: pd.DataFrame
) -> pd.DataFrame:
    assert_unique_key(sirh_df, "id_employee", "sirh_df")
    assert_unique_key(eval_df, "id_employee", "eval_df")
    assert_unique_key(sondage_df, "id_employee", "sondage_df")

    df = sirh_df.merge(eval_df, on="id_employee", how="inner")
    df = df.merge(sondage_df, on="id_employee", how="inner")

    if TARGET_COL in df.columns:
        df[TARGET_COL] = normalize_yes_no(df[TARGET_COL])

    return df

df = build_training_dataframe(sirh_df, eval_df, sondage_df)
display(df.head())
print(df.shape)

,id_employee,age,genre,revenu_mensuel,statut_marital,departement,poste,nombre_experiences_precedentes,nombre_heures_travailless,annee_experience_totale,...,nombre_participation_pee,nb_formations_suivies,nombre_employee_sous_responsabilite,distance_domicile_travail,niveau_education,domaine_etude,ayant_enfants,frequence_deplacement,annees_depuis_la_derniere_promotion,annes_sous_responsable_actuel
0,1,41,F,5993,Célibataire,Commercial,Cadre Commercial,8,80,8,...,0,0,1,1,2,Infra & Cloud,Y,Occasionnel,0,5
1,2,49,M,5130,Marié(e),Consulting,Assistant de Direction,1,80,10,...,1,3,1,8,1,Infra & Cloud,Y,Frequent,1,7
2,4,37,M,2090,Célibataire,Consulting,Consultant,6,80,7,...,0,3,1,2,2,Autre,Y,Occasionnel,0,0
3,5,33,F,2909,Marié(e),Consulting,Assistant de Direction,1,80,8,...,0,3,1,3,4,Infra & Cloud,Y,Frequent,3,0
4,7,27,M,3468,Marié(e),Consulting,Consultant,9,80,6,...,1,3,1,2,1,Transformation Digitale,Y,Occasionnel,2,2


(1470, 32)


In [45]:
print(df[TARGET_COL].value_counts(dropna=False))

a_quitte_l_entreprise
non    1233
oui     237
Name: count, dtype: int64


In [46]:
df[TARGET_COL] = (
    df[TARGET_COL]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({
        "yes": "oui",
        "no": "non",
        "1": "oui",
        "0": "non",
        "true": "oui",
        "false": "non"
    })
)

In [47]:
y = df[TARGET_COL].map({"oui": 1, "non": 0})
mask = y.notna()

df_model = df.loc[mask].copy()
y = y.loc[mask].astype(int)

print(df_model.shape)
print(y.value_counts())

(1470, 32)
a_quitte_l_entreprise
0    1233
1     237
Name: count, dtype: int64


In [48]:
cols_to_drop = [
    TARGET_COL,
    "id_employee"
]

In [49]:
X = df_model.drop(columns=cols_to_drop, errors="ignore").copy()
feature_columns = X.columns.tolist()

print(X.shape)
print(feature_columns)

(1470, 30)
['age', 'genre', 'revenu_mensuel', 'statut_marital', 'departement', 'poste', 'nombre_experiences_precedentes', 'nombre_heures_travailless', 'annee_experience_totale', 'annees_dans_l_entreprise', 'annees_dans_le_poste_actuel', 'satisfaction_employee_environnement', 'note_evaluation_precedente', 'niveau_hierarchique_poste', 'satisfaction_employee_nature_travail', 'satisfaction_employee_equipe', 'satisfaction_employee_equilibre_pro_perso', 'note_evaluation_actuelle', 'heure_supplementaires', 'augementation_salaire_precedente', 'nombre_participation_pee', 'nb_formations_suivies', 'nombre_employee_sous_responsabilite', 'distance_domicile_travail', 'niveau_education', 'domaine_etude', 'ayant_enfants', 'frequence_deplacement', 'annees_depuis_la_derniere_promotion', 'annes_sous_responsable_actuel']


Split Train / Test

In [50]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)
print("y_train :", y_train.shape)
print("y_test  :", y_test.shape)

X_train : (1176, 30)
X_test  : (294, 30)
y_train : (1176,)
y_test  : (294,)


Pipeline de preprocessing + modèle

In [51]:
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, selector(dtype_include=np.number)),
    ("cat", categorical_pipe, selector(dtype_exclude=np.number)),
])

pipe = Pipeline([
    ("prep", preprocessor),
    ("clf", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

Cross Val

In [52]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

In [53]:
cv_scores = cross_validate(
    pipe,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

In [54]:
cv_results = pd.DataFrame({
    metric: cv_scores[f"test_{metric}"]
    for metric in scoring.keys()
})

cv_results

,accuracy,precision,recall,f1,roc_auc
0,0.805085,0.431034,0.657895,0.520833,0.803828
1,0.753191,0.364865,0.710526,0.482143,0.793214
2,0.731915,0.356322,0.815789,0.496000,0.857868
3,0.753191,0.371795,0.763158,0.500000,0.867486
4,0.761702,0.378378,0.736842,0.500000,0.823270


In [55]:
cv_results.agg(["mean", "std"]).T

,mean,std
accuracy,0.761017,0.026981
precision,0.380479,0.029424
recall,0.736842,0.058844
f1,0.499795,0.013863
roc_auc,0.829133,0.032640


Entrainement final

In [56]:
pipe.fit(X_train, y_train)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x000001D2F50CFAD0>),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x000001D2F50CEE70>)])),
                ('clf',
                 LogisticRegression(class_weight='balanced', max_iter=2000,
                                    random_state=42))])

In [57]:
proba_test = pipe.predict_proba(X_test)[:, 1]
pred_test = (proba_test >= 0.5).astype(int)

In [58]:
metrics_test = {
    "accuracy": accuracy_score(y_test, pred_test),
    "precision": precision_score(y_test, pred_test, zero_division=0),
    "recall": recall_score(y_test, pred_test, zero_division=0),
    "f1": f1_score(y_test, pred_test, zero_division=0),
    "roc_auc": roc_auc_score(y_test, proba_test),
    "log_loss": log_loss(y_test, np.clip(proba_test, 1e-6, 1 - 1e-6))
}

metrics_test

{'accuracy': 0.7619047619047619,
 'precision': np.float64(0.37362637362637363),
 'recall': np.float64(0.723404255319149),
 'f1': np.float64(0.4927536231884058),
 'roc_auc': np.float64(0.8065294168317686),
 'log_loss': 0.4774575732785039}

In [59]:
print(confusion_matrix(y_test, pred_test))

[[190  57]
 [ 13  34]]


In [60]:
print(classification_report(y_test, pred_test, digits=3))

              precision    recall  f1-score   support

           0      0.936     0.769     0.844       247
           1      0.374     0.723     0.493        47

    accuracy                          0.762       294
   macro avg      0.655     0.746     0.669       294
weighted avg      0.846     0.762     0.788       294



In [61]:
proba_train = pipe.predict_proba(X_train)[:, 1]

precision, recall, thresholds = precision_recall_curve(y_train, proba_train)

thresholds = np.r_[0, thresholds]
f1_scores = 2 * precision * recall / (precision + recall + 1e-12)

best_idx = np.nanargmax(f1_scores)
best_threshold = float(thresholds[best_idx])

print("Best threshold :", best_threshold)
print("Best F1 train  :", f1_scores[best_idx])

Best threshold : 0.7671641395016716
Best F1 train  : 0.6344410876128039


In [62]:
pred_test_best = (proba_test >= best_threshold).astype(int)

In [63]:
metrics_test_best = {
    "accuracy": accuracy_score(y_test, pred_test_best),
    "precision": precision_score(y_test, pred_test_best, zero_division=0),
    "recall": recall_score(y_test, pred_test_best, zero_division=0),
    "f1": f1_score(y_test, pred_test_best, zero_division=0),
    "roc_auc": roc_auc_score(y_test, proba_test)
}

metrics_test_best

{'accuracy': 0.8571428571428571,
 'precision': np.float64(0.5806451612903226),
 'recall': np.float64(0.3829787234042553),
 'f1': np.float64(0.46153846153846156),
 'roc_auc': np.float64(0.8065294168317686)}

In [64]:
print(confusion_matrix(y_test, pred_test_best))
print(classification_report(y_test, pred_test_best, digits=3))

[[234  13]
 [ 29  18]]
              precision    recall  f1-score   support

           0      0.890     0.947     0.918       247
           1      0.581     0.383     0.462        47

    accuracy                          0.857       294
   macro avg      0.735     0.665     0.690       294
weighted avg      0.840     0.857     0.845       294



In [65]:
model_path = MODEL_DIR / MODEL_FILENAME
joblib.dump(pipe, model_path)

print(f"Modèle sauvegardé dans : {model_path}")

Modèle sauvegardé dans : artifacts\attrition_pipeline.joblib


In [66]:
metadata = {
    "model_name": "attrition_classifier",
    "model_type": "LogisticRegression",
    "target": TARGET_COL,
    "features": feature_columns,
    "threshold": best_threshold,
    "test_metrics_default_threshold": metrics_test,
    "test_metrics_best_threshold": metrics_test_best,
    "train_size": int(len(X_train)),
    "test_size": int(len(X_test)),
    "random_state": RANDOM_STATE
}

In [67]:
metadata_path = MODEL_DIR / METADATA_FILENAME

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print(f"Métadonnées sauvegardées dans : {metadata_path}")

Métadonnées sauvegardées dans : artifacts\model_metadata.json


Export d’un payload d’exemple pour l’API

In [68]:
sample_payload = X_test.head(3).copy()
sample_payload

,age,genre,revenu_mensuel,statut_marital,departement,poste,nombre_experiences_precedentes,nombre_heures_travailless,annee_experience_totale,annees_dans_l_entreprise,...,nombre_participation_pee,nb_formations_suivies,nombre_employee_sous_responsabilite,distance_domicile_travail,niveau_education,domaine_etude,ayant_enfants,frequence_deplacement,annees_depuis_la_derniere_promotion,annes_sous_responsable_actuel
1061,24,F,2033,Marié(e),Commercial,Représentant Commercial,1,80,1,1,...,1,2,1,13,2,Infra & Cloud,Y,Aucun,0,0
891,44,F,2011,Marié(e),Consulting,Assistant de Direction,1,80,10,10,...,1,5,1,2,1,Infra & Cloud,Y,Occasionnel,7,7
456,31,M,11557,Divorcé(e),Commercial,Senior Manager,9,80,10,5,...,1,3,1,7,3,Infra & Cloud,Y,Occasionnel,0,1


In [69]:
sample_payload_path = MODEL_DIR / "sample_payload.json"

sample_payload.to_json(
    sample_payload_path,
    orient="records",
    force_ascii=False,
    indent=2
)

print(f"Payload exemple sauvegardé dans : {sample_payload_path}")

Payload exemple sauvegardé dans : artifacts\sample_payload.json


Vérification du rechargement

In [70]:
loaded_model = joblib.load(model_path)

loaded_proba = loaded_model.predict_proba(sample_payload)[:, 1]
loaded_pred = (loaded_proba >= best_threshold).astype(int)

print("Probabilités :", loaded_proba)
print("Prédictions  :", loaded_pred)

Probabilités : [0.26320638 0.02273135 0.30304642]
Prédictions  : [0 0 0]


Sauvegarde d’un extrait de résultats

In [71]:
results_df = X_test.copy()
results_df["y_true"] = y_test.values
results_df["proba"] = proba_test
results_df["pred_0_5"] = pred_test
results_df["pred_best_threshold"] = pred_test_best

results_df.head()

,age,genre,revenu_mensuel,statut_marital,departement,poste,nombre_experiences_precedentes,nombre_heures_travailless,annee_experience_totale,annees_dans_l_entreprise,...,niveau_education,domaine_etude,ayant_enfants,frequence_deplacement,annees_depuis_la_derniere_promotion,annes_sous_responsable_actuel,y_true,proba,pred_0_5,pred_best_threshold
1061,24,F,2033,Marié(e),Commercial,Représentant Commercial,1,80,1,1,...,2,Infra & Cloud,Y,Aucun,0,0,0,0.263206,0,0
891,44,F,2011,Marié(e),Consulting,Assistant de Direction,1,80,10,10,...,1,Infra & Cloud,Y,Occasionnel,7,7,0,0.022731,0,0
456,31,M,11557,Divorcé(e),Commercial,Senior Manager,9,80,10,5,...,3,Infra & Cloud,Y,Occasionnel,0,1,0,0.303046,0,0
922,44,M,19190,Divorcé(e),Consulting,Senior Manager,1,80,26,25,...,2,Infra & Cloud,Y,Occasionnel,14,13,0,0.024812,0,0
69,36,M,3388,Marié(e),Consulting,Assistant de Direction,0,80,2,1,...,3,Transformation Digitale,Y,Occasionnel,0,0,1,0.694637,1,0


In [72]:
predictions_path = MODEL_DIR / "predictions_sample.csv"
results_df.head(50).to_csv(predictions_path, index=False)

print(f"Extrait de prédictions sauvegardé dans : {predictions_path}")

Extrait de prédictions sauvegardé dans : artifacts\predictions_sample.csv


In [73]:
print("Notebook terminé avec succès.")
print(f"Modèle        : {model_path}")
print(f"Métadonnées   : {metadata_path}")
print(f"Payload       : {sample_payload_path}")
print(f"Prédictions   : {predictions_path}")

Notebook terminé avec succès.
Modèle        : artifacts\attrition_pipeline.joblib
Métadonnées   : artifacts\model_metadata.json
Payload       : artifacts\sample_payload.json
Prédictions   : artifacts\predictions_sample.csv


In [74]:
from pathlib import Path

print("Répertoire courant :", Path().resolve())
print("MODEL_DIR brut     :", Path("../artifacts").resolve())

Répertoire courant : C:\Users\mouto\Documents\Jedha\OCR_Projects\OCR_Projet_5
MODEL_DIR brut     : C:\Users\mouto\Documents\Jedha\OCR_Projects\artifacts
